# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (`data.csv`), dictionary in `columns.csv`.

Plan: inspect data, check for leakage, build features available at snapshot time, evaluate with splits that respect repeated accounts and time, and compare against naive baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data overview

In [ ]:
print(df.shape, '| accounts:', df.account_ref.nunique(), '| nulls:', df.isna().sum().sum(),
      '| duplicate account-months:', df.duplicated(['account_ref', 'snapshot_month']).sum())
print('churn rate:', round(df.churn_next_month.mean(), 4))
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby('snapshot_month').churn_next_month.mean().plot(ax=ax[0], marker='o')
ax[0].set_title('Churn rate by snapshot month'); ax[0].tick_params(axis='x', rotation=90)
df.groupby('plan').churn_next_month.mean().plot.bar(ax=ax[1]); ax[1].set_title('Churn rate by plan')
plt.tight_layout()

About 15% of rows churn, with ~5 snapshots per account. Churn varies strongly by calendar month (about 3% in spring 2024 vs 50%+ in Dec 2024 to Jan 2025), and barely by plan.

## 2. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export on 2026-01-05*, so it is only known after the fact. Compare it with `tenure_months`:

In [ ]:
df['lifetime_minus_tenure'] = df.lifetime_months - df.tenure_months
print(df.groupby('churn_next_month').lifetime_minus_tenure.describe())
print('AUC of -lifetime_months alone:', round(roc_auc_score(df.churn_next_month, -df.lifetime_months), 3))
print('AUC of (lifetime == tenure) flag:', round(roc_auc_score(df.churn_next_month, df.lifetime_minus_tenure == 0), 3))

Every churned row has `lifetime_months == tenure_months` (the account's active life ends at the snapshot) and no non-churned row does, so the column encodes the label from the future. It would give a near-perfect score here and not be available when making a real prediction. **It is excluded from every model.** `snapshot_month` is also not used as a feature, since calendar effects can't be extrapolated to future months.

## 3. Features (only information known at snapshot time)
Current-month fields, per-seat ratios, and, from the same account's *earlier* snapshot only, the change in usage and prior tickets (when the previous month exists).

In [ ]:
d = df.sort_values(['account_ref', 'snapshot_month']).reset_index(drop=True)
d['period'] = pd.PeriodIndex(d.snapshot_month, freq='M').astype(int)
g = d.groupby('account_ref')
consec = (d.period - g.period.shift(1)) == 1
d['usage_prev'] = g.usage_hours.shift(1).where(consec)
d['usage_change'] = d.usage_hours - d.usage_prev
d['usage_ratio'] = (d.usage_hours / d.usage_prev).replace(np.inf, np.nan)  # prev usage of 0 -> undefined
d['tickets_prev'] = g.support_tickets.shift(1).where(consec)
d['fee_per_seat'] = d.monthly_fee / d.seats
d['usage_per_seat'] = d.usage_hours / d.seats
cat = ['plan']
base = ['seats', 'monthly_fee', 'fee_per_seat', 'tenure_months', 'usage_hours', 'usage_per_seat', 'support_tickets']
hist = ['usage_change', 'usage_ratio', 'tickets_prev']
y = d.churn_next_month.values; groups = d.account_ref.values
print('missing share:', d[base + hist].isna().mean().round(3).to_dict())
d.groupby(pd.qcut(d.usage_hours, 8)).churn_next_month.mean().plot.bar(title='Churn rate by usage-hours octile', figsize=(7, 3));

## 4. Models and evaluation
Accounts appear in many rows, so random row splits would leak account identity between train and test. We use **5-fold GroupKFold by account**. Metrics: ROC-AUC and PR-AUC (base rate is about 0.15, so an uninformative model has PR-AUC of about 0.15).

In [ ]:
def gb():
    return HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, l2_regularization=1.0, random_state=0)
def lr(num):
    ct = ColumnTransformer([('c', OneHotEncoder(), cat),
                            ('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num)])
    return make_pipeline(ct, LogisticRegression(max_iter=2000))
def gbp(num):
    return make_pipeline(ColumnTransformer([('c', OneHotEncoder(), cat), ('n', 'passthrough', num)]), gb())
def cv(make, cols):
    oof = np.zeros(len(d))
    for tr, te in GroupKFold(5).split(d, y, groups):
        m = make().fit(d.iloc[tr][cols], y[tr]); oof[te] = m.predict_proba(d.iloc[te][cols])[:, 1]
    return oof
res, oofs = {}, {}
for name, make, cols in [('Logistic (current-month)', lambda: lr(base), cat + base),
                         ('GBM (current-month)', lambda: gbp(base), cat + base),
                         ('Logistic (+history)', lambda: lr(base + hist), cat + base + hist),
                         ('GBM (+history)', lambda: gbp(base + hist), cat + base + hist)]:
    o = cv(make, cols); oofs[name] = o
    res[name] = (roc_auc_score(y, o), average_precision_score(y, o))
res['Baseline: -usage_hours'] = (roc_auc_score(y, -d.usage_hours), average_precision_score(y, -d.usage_hours))
res['Baseline: -tenure_months'] = (roc_auc_score(y, -d.tenure_months), average_precision_score(y, -d.tenure_months))
res['Baseline: constant'] = (0.5, y.mean())
pd.DataFrame(res, index=['ROC-AUC', 'PR-AUC']).T.round(3)

## 5. Does it hold up over time?
Train on snapshots up to 2025-03, test on 2025-05 onward (one-month gap), dropping test accounts that also appear in training.

In [ ]:
cols = cat + base + hist
tr_mask = (d.snapshot_month <= '2025-03').values
te_mask = (d.snapshot_month >= '2025-05').values & ~d.account_ref.isin(d.account_ref[tr_mask]).values
print('train', tr_mask.sum(), 'test', te_mask.sum(), '| churn rate train/test:', y[tr_mask].mean().round(3), y[te_mask].mean().round(3))
rows = []
for name, make in [('Logistic', lambda: lr(base + hist)), ('GBM', lambda: gbp(base + hist))]:
    m = make().fit(d[tr_mask][cols], y[tr_mask]); p = m.predict_proba(d[te_mask][cols])[:, 1]
    rows.append((name, roc_auc_score(y[te_mask], p), average_precision_score(y[te_mask], p)))
pd.DataFrame(rows, columns=['model', 'ROC-AUC', 'PR-AUC']).round(3)

## 6. What drives the prediction, and is the risk score calibrated?

In [ ]:
tr, te = next(GroupKFold(5).split(d, y, groups))
m = gbp(base + hist).fit(d.iloc[tr][cols], y[tr])
pi = permutation_importance(m, d.iloc[te][cols], y[te], scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=cols).sort_values().plot.barh(title='Permutation importance (drop in ROC-AUC)', figsize=(7, 4));

In [ ]:
o = oofs['GBM (+history)']
dec = pd.qcut(o, 10, labels=False, duplicates='drop')
t = pd.DataFrame({'decile': dec, 'y': y, 'p': o}).groupby('decile').agg(observed=('y', 'mean'), predicted=('p', 'mean'), n=('y', 'size'))
print(t.round(3))
t[['observed', 'predicted']].plot.bar(title='Observed vs predicted churn by risk decile (out-of-fold)', figsize=(7, 3));

## 7. Conclusions
- **Yes, but only modestly.** Grouped 5-fold CV gives ROC-AUC of about 0.67 and PR-AUC of about 0.29 (base rate 0.154). The best single-feature baseline, low usage, reaches 0.61 AUC; tenure alone is uninformative.
- **Risk ranking is usable, the individual prediction is weak.** Predicted risk is well calibrated across deciles: the top decile churns at about 36% (about 2.4x the base rate) and the bottom two at about 7-8%. That is enough to prioritise outreach, not to call individual cancellations.
- **Leakage matters most.** `lifetime_months` is a post-hoc field that equals `tenure_months` for exactly the churned rows (AUC 1.0). It is excluded; any model using it would look excellent and be useless in production.
- **Extra history features add nothing** (GBM 0.675 with or without them), and gradient boosting is barely better than logistic regression, so a simple model is sufficient.
- **Time shift is the main caveat.** Churn rate differs a lot by calendar month (about 3% to 55%), and the later test window has 10% churn vs 19% in training. Trained on early months and tested on later, unseen accounts, the logistic model still reaches AUC of about 0.70 (PR-AUC 0.26 vs a 0.10 base rate), but the GBM drops to about 0.67. The month-level swings are not explained by these features, and would need external drivers (pricing changes, campaigns, seasonality) to forecast.
- **Next steps:** find out what drove the Nov 2024 to Jan 2025 churn wave, add features such as billing, contract and payment-failure data, and check the first/last snapshot per account for survivorship effects.

Note: the stored outputs were produced by running the cells in a plain Python process (the sandbox disallowed starting a Jupyter kernel); re-run in Jupyter to refresh them.